# UC1 Results — FedGen variant analysis

Two organisational axes:

| Axis | Levels | Question answered |
|---|---|---|
| **Data case** | filtered / unfiltered | Does enforcing min-patients & min-positive constraints change conclusions? |
| **Sharing family** | partial / full | What is the privacy cost? (partial keeps θ_f local) |

Within each family, three ablation dimensions are crossed:
- **KD loss**: hard CE vs KL ensemble (Zhu Eq. 5)
- **Prototype anchor**: centroid vs medoid (clinical traceability)
- **Generator**: neural G_ω vs GMMSampler (capacity ablation)

### Evaluation protocols

- **Full-sharing** → single global model evaluated on concatenated test splits (`[G]`)
- **Partial-sharing** → each client uses its own local encoder + global predictor, evaluated on its own local test split (`[P]`)

Comparing AUC numbers across families is **not apples-to-apples**.
The meaningful comparison is the *privacy gap*: how much accuracy does each
partial variant lose relative to its full counterpart?

## Setup

In [ ]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, '..')
from UC1PrintingUtils import (
    load_results,
    print_results_table,
    plot_convergence,
    plot_pareto,
    plot_equity,
    plot_auc_vs_alpha,
    plot_auc_vs_alpha_families,
    plot_partial_vs_full_band,
    plot_centroid_vs_medoid,
    plot_equity_weighted,
    print_equity_table,
    
)

FEDAVG_RESULTS  = '../02_FedAvg/results'
FEDGEN_RESULTS  = '../03_FedGen/results'
ALPHA_SWEEP     = [0.1, 0.5, 1.0, 5.0, 10.0]
SEEDS           = [42, 123, 7]
CENTRALIZED_AUC = 0.658

VARIANTS_PARTIAL = {
    'fedavg_partial':        {'label': 'FedAvg (partial)',         'color': '#d62728', 'ls': '-',  'marker': 'o'},
    'fedgen_partial':        {'label': 'FedGen (partial)(centroid)',         'color': '#ff7f0e', 'ls': '--', 'marker': 's'},
    'fedgen_partial_medoid': {'label': 'FedGen partial (medoid)', 'color': '#2ca02c', 'ls': '--', 'marker': 's'},
    'fedgen_zhu':            {'label': 'FedGen-Zhu (KL)(centroid)',         'color': '#e377c2', 'ls': '-',  'marker': '^'},
    'fedgen_zhu_medoid':     {'label': 'FedGen-Zhu (KL, medoid)','color': '#17becf', 'ls': '-',  'marker': '^'},
    'fedgen_zhu_no_proto':   {'label': 'FedGen-Zhu (no proto)',   'color': '#ff7f0e', 'ls': '--', 'marker': '^'},
    'fedgen_gmm':            {'label': 'FedGen-GMM (partial)',    'color': '#9467bd', 'ls': '--', 'marker': 'D'},
}
VARIANTS_FULL = {
    'fedavg_full':            {'label': 'FedAvg (full)',            'color': '#1f77b4', 'ls': '-',  'marker': 'o'},
    'fedgen_full':            {'label': 'FedGen (full)(centroid)',            'color': '#6baed6', 'ls': '--', 'marker': 's'},
    'fedgen_full_medoid':     {'label': 'FedGen full (medoid)',     'color': '#2ca02c', 'ls': '-',  'marker': 's'},
    'fedgen_zhu_full':        {'label': 'FedGen-Zhu (full)(centroid)',        'color': '#bcbd22', 'ls': '-',  'marker': 'D'},
    'fedgen_zhu_full_medoid': {'label': 'FedGen-Zhu full (medoid)','color': '#8c564b', 'ls': '-',  'marker': '^'},
    'fedgen_zhu_full_no_proto':{'label': 'FedGen-Zhu (full, no proto)', 'color': '#6baed6', 'ls': '--', 'marker': '^'},
    'fedgen_gmm_full':        {'label': 'FedGen-GMM (full)',        'color': '#9467bd', 'ls': '-',  'marker': 'D'},
}
VARIANTS_ALL = {**VARIANTS_PARTIAL, **VARIANTS_FULL}

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})
os.makedirs('figures', exist_ok=True)
print('Setup complete.')

## Load results

In [ ]:
all_results = {}
for data_case in ['filtered', 'unfiltered']:
    all_results[data_case] = load_results(
        FEDAVG_RESULTS, FEDGEN_RESULTS,
        ALPHA_SWEEP, SEEDS, VARIANTS_ALL,
        data_case=data_case,
    )

---
## 1. Summary tables

Using `print_results_table` from UC1PrintingUtils, called per family.

In [ ]:
for dc in ['filtered', 'unfiltered']:
    print(f'\n{"#"*70}\n  {dc.upper()} — PARTIAL FAMILY [P]\n{"#"*70}')
    print_results_table(all_results[dc], ALPHA_SWEEP, VARIANTS_PARTIAL, dc)
    print(f'\n{"#"*70}\n  {dc.upper()} — FULL FAMILY [G]\n{"#"*70}')
    print_results_table(all_results[dc], ALPHA_SWEEP, VARIANTS_FULL, dc)

---
## 2. Convergence curves — by family

Test AUC vs communication round.

In [ ]:
for dc in ['filtered', 'unfiltered']:
    plot_convergence(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, VARIANTS_PARTIAL, data_case=f'{dc}_partial')
    plot_convergence(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, VARIANTS_FULL,    data_case=f'{dc}_full')

---
## 3. AUC vs heterogeneity — two-panel (partial | full)

In [ ]:
for dc in ['filtered', 'unfiltered']:
    plot_auc_vs_alpha_families(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, dc)

---
## 4. Pareto frontier — by family

AUC vs cumulative MB.

In [ ]:
for dc in ['filtered', 'unfiltered']:
    plot_pareto(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, VARIANTS_PARTIAL, data_case=f'{dc}_partial')
    plot_pareto(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, VARIANTS_FULL,    data_case=f'{dc}_full')

---
## 5. Partial vs full-sharing — privacy gap

Grey band = range across all full-sharing variants (accuracy ceiling).
Coloured lines = partial-sharing variants.

In [ ]:
for dc in ['filtered', 'unfiltered']:
    plot_partial_vs_full_band(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, dc)

---
## 6. Centroid vs medoid — prototype anchor ablation

Paired bars across both families.

In [ ]:
for dc in ['filtered', 'unfiltered']:
    plot_centroid_vs_medoid(all_results[dc], ALPHA_SWEEP, dc)

---
## 7. Per-client equity — basic (from UC1PrintingUtils)

Bar chart with jittered dots. Called per family.

In [ ]:
for dc in ['filtered', 'unfiltered']:
    plot_equity(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, VARIANTS_PARTIAL, data_case=f'{dc}_partial')
    plot_equity(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, VARIANTS_FULL,    data_case=f'{dc}_full')

---
## 8. Per-client equity — weighted vs unweighted (partial only)

- **Weighted mean**: large hospitals count more
- **Unweighted mean**: each hospital equally
- **Gap**: weighted − unweighted (positive = large hospitals favoured)
- **Min client AUC**: worst-served hospital
- **Spread**: max − min

In [ ]:


for dc in ['filtered', 'unfiltered']:
    plot_equity_weighted(all_results[dc], ALPHA_SWEEP, CENTRALIZED_AUC, dc)
    print_equity_table(all_results[dc], ALPHA_SWEEP, dc)

---
## 9. Export CSV

In [ ]:
for data_case in ['filtered', 'unfiltered']:
    results = all_results[data_case]
    rows = []
    for alpha in ALPHA_SWEEP:
        for vname, vinfo in VARIANTS_ALL.items():
            d = results[alpha].get(vname)
            if not d or not d['test_aucs']: continue
            mbs = [m[-1] for m in d['cumul_mbs']] if d['cumul_mbs'] else [float('nan')]
            family = 'partial' if vname in VARIANTS_PARTIAL else 'full'
            rows.append({
                'data_case':   data_case,
                'family':      family,
                'alpha':       alpha,
                'variant':     vname,
                'label':       vinfo['label'],
                'auc_mean':    round(np.mean(d['test_aucs']), 4),
                'auc_std':     round(np.std(d['test_aucs']),  4),
                'mb_mean':     round(np.mean(mbs), 2),
                'rounds_mean': round(np.mean([len(h) for h in d['histories']]), 1),
            })
    path = f'figures/results_{data_case}.csv'
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f'Saved → {path}  ({len(rows)} rows)')